In [1]:
import ast
from collections import Counter
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import Normalizer
import warnings

warnings.filterwarnings('ignore')

main_data_path = '../../pickle-object/ttrpg_bgg_description_svd_table.csv'
ref_data_path = 'cluster_reference_data.csv'
output_txt = 'cluster_profiles_summary.txt'

main_df = pd.read_csv(main_data_path, low_memory=False)
ref_df = pd.read_csv(ref_data_path)

svd_cols = [col for col in ref_df.columns if str(col).startswith('SV_')]

for col in svd_cols:
    main_df[col] = pd.to_numeric(main_df[col], errors='coerce').fillna(0)

X_main = main_df[svd_cols].values
centroids = ref_df[svd_cols].values
X_scaled = Normalizer(norm='l2').fit_transform(X_main)

distances = cdist(X_scaled, centroids, metric='euclidean')

cluster_mapping = ref_df['Cluster_ID'].values if 'Cluster_ID' in ref_df.columns else ref_df.index.values
main_df['Cluster_ID'] = [cluster_mapping[idx] for idx in np.argmin(distances, axis=1)]

def safe_parse_list(val):
    if pd.isna(val):
        return []
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return parsed
        return []
    except (ValueError, SyntaxError):
        return [x.strip() for x in str(val).split(',') if x.strip()]

def get_top_features(feature_series, top_n=3):
    all_items = []
    for item_list in feature_series:
        all_items.extend(item_list)
    if not all_items:
        return "None"
    most_common = Counter(all_items).most_common(top_n)
    # Extracts only the name, leaving out the frequency counts
    return ", ".join([item for item, count in most_common])

main_df['Category_List'] = main_df['Category'].apply(safe_parse_list)
main_df['Mechanics_List'] = main_df['Mechanics'].apply(safe_parse_list)

# value_counts() automatically sorts from highest to lowest
cluster_sizes = main_df['Cluster_ID'].value_counts()

with open(output_txt, 'w', encoding='utf-8') as f:
    for c_id, size in cluster_sizes.items():
        cluster_data = main_df[main_df['Cluster_ID'] == c_id]
        
        top_cats = get_top_features(cluster_data['Category_List'], top_n=3)
        top_mechs = get_top_features(cluster_data['Mechanics_List'], top_n=3)
        
        f.write(f"Cluster {int(c_id)} | {size} games | {top_cats} | {top_mechs} |\n")

print(f"Report successfully saved to '{output_txt}'.")

Report successfully saved to 'cluster_profiles_summary.txt'.
